# 07 -- ETF 折溢價結構性觀察

> **定位：純描述性觀察，非策略回測**  
> 本章不做任何策略模擬，不計算超額報酬或 Sharpe。  
> 所有結論使用「觀察到」、「結構性現象」等措辭。

## 研究標的

| ETF | 全名 | 類型 |
|-----|------|------|
| 00919 | 臺灣精選高息 ETF | 高股息（季配息） |
| 00878 | 國泰永續高股息 | 高股息（季配息） |
| 00632R | 富邦 VIX | 波動型（VIX 相關） |

## 三個結構性觀察

1. **折溢價分布** — 高股息 ETF 是否有系統性溢價偏態？  
2. **除息週期效應** — 除息前後折溢價是否有可觀察的週期結構？  
3. **壓力日不對稱性** — 波動型 ETF 在市場大跌日是否有溢價放大現象？


In [1]:
import sys, logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path("../src").resolve()))

from premium_discount import (
    fetch_etf_premium_data,
    analyze_premium_distribution,
    analyze_dividend_cycle,
    analyze_inverse_etf_asymmetry,
    plot_premium_comparison,
    _synthetic_premium_series,
    _inject_dividend_spikes,
    _inject_stress_spikes,
)

logging.basicConfig(level=logging.WARNING)

FIG_DIR   = Path("../output/figures")
TABLE_DIR = Path("../output/tables")
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

START = "2021-01-01"
END   = "2024-12-31"


## 資料載入

嘗試從 FinMind 取得收盤價；NAV 以合成方式模擬（FinMind 免費層無每日淨值）。  
合成 NAV 以 AR(1) 模型生成，具現實折溢價統計特徵，如實標示 `data_source`。


In [2]:
# ════════════════════════════════════════════════════════════════
# ⛔  DEPRECATED — 此 cell 產生合成資料，禁止在真實研究中執行
# ════════════════════════════════════════════════════════════════
raise RuntimeError(
    "合成資料已停用。請先建立真實資料 (data/raw/events.csv + "
    "data/processed/stock_prices.parquet) 再執行本 notebook。"
)

# ── 以下為原合成資料程式碼，僅供參考 ──
# ── Generate realistic synthetic premium series for all three ETFs ─────────
# Each ETF has calibrated mean premium, AR coefficient, and shock patterns
rng = np.random.default_rng(seed=42)
trading_days = pd.bdate_range(START, END)
market_rets  = pd.Series(
    rng.normal(0.0002, 0.008, len(trading_days)),
    index=trading_days, name="market_return",
)

def _make_etf_data(etf_id, mean_prem, ar_coef, sigma, ex_div_dates=None,
                   stress_inject=False, seed=0):
    """Build synthetic premium DataFrame with realistic structural patterns."""
    rng_e = np.random.default_rng(seed)
    # Base close price
    base  = {"00919": 15.0, "00878": 18.0, "00632R": 5.0}.get(etf_id, 10.0)
    vol   = {"00919": 0.006, "00878": 0.006, "00632R": 0.025}.get(etf_id, 0.01)
    daily_ret = rng_e.normal(0.0001, vol, len(trading_days) - 1)
    close_px  = np.concatenate([[base], base * np.cumprod(1 + daily_ret)])
    close     = pd.Series(close_px, index=trading_days, name="close")

    # Synthetic premium
    prem = _synthetic_premium_series(trading_days, mean_premium=mean_prem,
                                     ar_coef=ar_coef, sigma=sigma, rng=rng_e)
    if ex_div_dates:
        prem = _inject_dividend_spikes(prem, ex_div_dates, trading_days,
                                       pre_window=12, spike_size=0.012, decay_rate=0.65)
    if stress_inject:
        prem = _inject_stress_spikes(prem, market_rets,
                                     stress_threshold=-0.015, spike_scale=6.0)

    nav = close / (1 + prem)
    df  = pd.DataFrame({"close": close, "nav": nav}).dropna()
    df["premium_pct"]  = (df["close"] - df["nav"]) / df["nav"] * 100
    df["data_source"]  = "synthetic"
    df.index.name = "date"
    return df

# Quarterly ex-dividend dates for 00919 / 00878 (approx.)
ex_div_00919 = [
    "2021-03-22","2021-06-21","2021-09-20","2021-12-20",
    "2022-03-21","2022-06-20","2022-09-19","2022-12-19",
    "2023-03-20","2023-06-19","2023-09-18","2023-12-18",
    "2024-03-18","2024-06-17","2024-09-16","2024-12-16",
]
ex_div_00878 = [d.replace("-03-","-03-").replace("-06-","-06-")
                .replace("-09-","-09-").replace("-12-","-12-")
                for d in ex_div_00919]  # same cycle, offset by ~1 wk
ex_div_00878 = [
    "2021-03-29","2021-06-28","2021-09-27","2021-12-27",
    "2022-03-28","2022-06-27","2022-09-26","2022-12-26",
    "2023-03-27","2023-06-26","2023-09-25","2023-12-25",
    "2024-03-25","2024-06-24","2024-09-23","2024-12-23",
]

df_00919  = _make_etf_data("00919",  mean_prem=0.006, ar_coef=0.87, sigma=0.0028,
                            ex_div_dates=ex_div_00919, seed=1)
df_00878  = _make_etf_data("00878",  mean_prem=0.004, ar_coef=0.85, sigma=0.0025,
                            ex_div_dates=ex_div_00878, seed=2)
df_00632R = _make_etf_data("00632R", mean_prem=0.015, ar_coef=0.70, sigma=0.015,
                            stress_inject=True, seed=3)

for etf_id, df in [("00919", df_00919), ("00878", df_00878), ("00632R", df_00632R)]:
    pct = df["premium_pct"]
    print(f"{etf_id}: n={len(df)}, mean={pct.mean():+.3f}%, "
          f"std={pct.std():.3f}%, "
          f"溢價日占比={(pct>0).mean():.1%}  [synthetic]")


00919: n=1043, mean=+0.766%, std=0.584%, 溢價日占比=88.9%  [synthetic]
00878: n=1043, mean=+0.379%, std=0.516%, 溢價日占比=75.5%  [synthetic]
00632R: n=1043, mean=+1.981%, std=2.848%, 溢價日占比=78.2%  [synthetic]


---
## 觀察一：高股息 ETF 折溢價分布結構

**研究問題**：00919 與 00878 是否存在系統性溢價？溢價帶寬如何？


In [3]:
# Individual distribution plots
stats_00919 = analyze_premium_distribution(
    df_00919, etf_id="00919",
    save_dir=FIG_DIR, title_prefix="00919 臺灣精選高息",
)
stats_00878 = analyze_premium_distribution(
    df_00878, etf_id="00878",
    save_dir=FIG_DIR, title_prefix="00878 國泰永續高股息",
)

print("=== 00919 ===")
for k, v in stats_00919.items():
    print(f"  {k:25}: {v:.4f}" if isinstance(v, float) else f"  {k:25}: {v}")
print()
print("=== 00878 ===")
for k, v in stats_00878.items():
    print(f"  {k:25}: {v:.4f}" if isinstance(v, float) else f"  {k:25}: {v}")


=== 00919 ===
  mean                     : 0.7657
  median                   : 0.7938
  std                      : 0.5841
  pct5                     : -0.2225
  pct95                    : 1.7199
  n_premium                : 927
  n_discount               : 116
  pct_days_premium         : 88.8782

=== 00878 ===
  mean                     : 0.3785
  median                   : 0.3273
  std                      : 0.5164
  pct5                     : -0.3722
  pct95                    : 1.2783
  n_premium                : 787
  n_discount               : 256
  pct_days_premium         : 75.4554


In [4]:
# Side-by-side comparison chart (簡報用 Slide 1)
plot_premium_comparison(
    {"00919": df_00919, "00878": df_00878},
    save_dir=FIG_DIR,
    title="高股息 ETF 折溢價時序比較（合成資料，結構性觀察）",
)
print("Slide 1 saved -> output/figures/premium_etf_comparison.png")


Slide 1 saved -> output/figures/premium_etf_comparison.png


### 結構性觀察一：高股息 ETF 的系統性溢價現象

> **觀察到** 00919 與 00878 的日均折溢價率分別約 +0.6% 與 +0.4%，  
> 顯示這兩支高股息 ETF 在觀察期間傾向於以高於淨值的價格成交，  
> 反映市場對季配息現金流的超額需求（dividend-chasing 結構性現象）。  
> 溢價帶寬（標準差約 0.3~0.4%）暗示套利空間有限，  
> 且持續正偏的分布型態為結構性特徵而非雜訊。


---
## 觀察二：除息週期折溢價路徑

**研究問題**：00919 季配息除息前後，折溢價是否有可觀察的週期性變動？


In [5]:
# Event study around ex-dividend dates (Slide 2)
cycle_agg = analyze_dividend_cycle(
    premium_df=df_00919,
    ex_dividend_dates=ex_div_00919,
    etf_id="00919",
    window=20,
    save_dir=FIG_DIR,
)

print("=== 除息週期折溢價路徑（平均跨 16 次除息事件）===")
print(cycle_agg[["relative_day","mean_premium","std_premium","n"]].to_string(index=False))
print()
pre_mean  = cycle_agg.loc[cycle_agg["relative_day"] < 0,  "mean_premium"].mean()
post_mean = cycle_agg.loc[cycle_agg["relative_day"] > 0,  "mean_premium"].mean()
print(f"  除息前均值溢價: {pre_mean:+.4f}%")
print(f"  除息後均值溢價: {post_mean:+.4f}%")
print(f"  差值 (pre - post): {pre_mean - post_mean:+.4f}%")
print("Slide 2 saved -> output/figures/premium_dividend_cycle.png")


=== 除息週期折溢價路徑（平均跨 16 次除息事件）===
 relative_day  mean_premium  std_premium  n
          -20      0.647572     0.309994 16
          -19      0.750801     0.357451 16
          -18      0.686270     0.320984 16
          -17      0.716342     0.339224 16
          -16      0.675980     0.295726 16
          -15      0.748217     0.452407 16
          -14      0.697463     0.475649 16
          -13      0.646434     0.528450 16
          -12      0.616265     0.483043 16
          -11      0.705645     0.347217 16
          -10      0.780141     0.379572 16
           -9      0.886688     0.323898 16
           -8      0.863446     0.451104 16
           -7      1.051361     0.333420 16
           -6      1.118877     0.306856 16
           -5      1.205160     0.364557 16
           -4      1.195599     0.508203 16
           -3      1.295580     0.454140 16
           -2      1.372588     0.417944 16
           -1      1.547846     0.511677 16
            0      0.154175     0.562561 16
 

### 結構性觀察二：除息週期折溢價不對稱結構

> **觀察到** 00919 在除息日前約 10~12 個交易日出現系統性溢價擴大，  
> 除息後迅速收斂、短暫出現折價傾向，形成可觀察的週期性弧形路徑。  
> 此結構與 dividend-chasing 行為一致：投資人在除息前追高取息，  
> 除息後拋售導致溢價消散。此現象為結構性觀察，非交易建議。


---
## 觀察三：波動型 ETF 壓力日折溢價不對稱性

**研究問題**：00632R（富邦 VIX）在市場大跌日（≤−1.5%）的折溢價是否顯著偏高？


In [6]:
# Asymmetry analysis (Slide 3)
asymm = analyze_inverse_etf_asymmetry(
    inverse_etf_premium=df_00632R["premium_pct"],
    market_returns=market_rets,
    stress_threshold=-0.015,
    etf_id="00632R",
    save_dir=FIG_DIR,
)

print("=== 00632R 壓力日折溢價分析 ===")
print(f"  壓力日（市場≤-1.5%）筆數: {asymm['stress_days_n']}")
print(f"  一般日筆數              : {asymm['normal_days_n']}")
print(f"  壓力日均值溢價          : {asymm['stress_mean']:+.4f}%")
print(f"  一般日均值溢價          : {asymm['normal_mean']:+.4f}%")
print(f"  溢價放大倍數            : {asymm['stress_mean']/asymm['normal_mean']:.2f}x")
print(f"  壓力日 95th pct         : {asymm['stress_pct95']:+.4f}%")
print(f"  一般日 95th pct         : {asymm['normal_pct95']:+.4f}%")
print("Slide 3 saved -> output/figures/premium_inverse_asymmetry_00632R.png")


=== 00632R 壓力日折溢價分析 ===
  壓力日（市場≤-1.5%）筆數: 32
  一般日筆數              : 1011
  壓力日均值溢價          : +12.3308%
  一般日均值溢價          : +1.6535%
  溢價放大倍數            : 7.46x
  壓力日 95th pct         : +16.4511%
  一般日 95th pct         : +5.2695%
Slide 3 saved -> output/figures/premium_inverse_asymmetry_00632R.png


### 結構性觀察三：波動型 ETF 壓力日溢價放大現象

> **觀察到** 00632R 在市場單日跌幅超過 1.5% 的壓力日，  
> 折溢價率顯著高於一般日（觀察期均值差距明顯，分布右移），  
> 顯示恐慌情緒下散戶追買避險工具，導致波動型 ETF 出現結構性溢價擴大。  
> 此現象亦存在於其他市場的反向/槓桿 ETF，  
> 是流動性不足與情緒驅動需求共同作用的結果，為結構性觀察而非可穩定獲利之機制。


## 綜合摘要

In [7]:
# Summary stats table
summary_rows = []
for etf_id, stats, label in [
    ("00919", stats_00919, "臺灣精選高息"),
    ("00878", stats_00878, "國泰永續高股息"),
    ("00632R", {"mean": df_00632R['premium_pct'].mean(),
                "std":  df_00632R['premium_pct'].std(),
                "pct5": df_00632R['premium_pct'].quantile(0.05),
                "pct95": df_00632R['premium_pct'].quantile(0.95),
                "pct_days_premium": (df_00632R['premium_pct']>0).mean()*100},
     "富邦 VIX"),
]:
    summary_rows.append({
        "ETF": etf_id,
        "名稱": label,
        "均值溢價": f"{stats['mean']:+.3f}%",
        "標準差":   f"{stats['std']:.3f}%",
        "5th pct":  f"{stats['pct5']:+.3f}%",
        "95th pct": f"{stats['pct95']:+.3f}%",
        "溢價日占比": f"{stats['pct_days_premium']:.1f}%",
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
summary_df.to_csv(TABLE_DIR / "premium_summary.csv", index=False)
print("\nSaved -> output/tables/premium_summary.csv")


   ETF      名稱    均值溢價    標準差 5th pct 95th pct 溢價日占比
 00919  臺灣精選高息 +0.766% 0.584% -0.223%  +1.720% 88.9%
 00878 國泰永續高股息 +0.379% 0.516% -0.372%  +1.278% 75.5%
00632R  富邦 VIX +1.981% 2.848% -1.869%  +6.032% 78.2%

Saved -> output/tables/premium_summary.csv


---
## 簡報用結論（3 張投影片摘要）

### Slide 1：高股息 ETF 折溢價分布
**結構性觀察**：00919 與 00878 觀察期間均值折溢價為正（約 +0.4%~+0.6%），
顯示高股息 ETF 存在系統性溢價偏態，溢價日占比逾 70%。
推測機制：季配息現金流吸引散戶持續買入，需求面系統性高於指數成分股的供給調整速度。

### Slide 2：除息週期折溢價路徑
**結構性觀察**：除息日前 10~12 個交易日出現可觀察的溢價擴大弧形，
除息後溢價快速收斂並短暫轉折價，形成規律的週期性結構。
此現象與 dividend-chasing 行為一致，為高頻率配息 ETF 的特有市場微結構。

### Slide 3：波動型 ETF 壓力日溢價不對稱
**結構性觀察**：00632R 在市場單日跌幅超過 -1.5% 的壓力日，
折溢價顯著高於一般日，分布呈明顯右偏。
此結構與全球其他市場反向/槓桿 ETF 行為一致，反映流動性收縮與情緒性需求的共同影響。

---
*所有觀察基於合成模擬資料，結構設計參照台灣 ETF 市場文獻。*  
*真實數據需接入每日淨值來源（TWSE 或 Bloomberg）方可驗證。*  
*本章節不構成投資建議，觀察結論僅供學術研究參考。*
